In [4]:
import math
import folium
import pandas as pd


# 1. 哈維辛公式：計算兩個經緯度之間的直線距離（公里）
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # 地球半徑 (km)

    # 將角度轉為弧度
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    # 運算公式
    a = (
        math.sin(delta_phi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c  # 傳回公里數


# 2. 智慧搜尋核心函式
def search_cafes(user_lat, user_lng, selected_tags, max_distance_km=1.0):
    """
    user_lat, user_lng: 使用者目前位置的經緯度
    selected_tags: 使用者勾選的標籤清單，例如 ['pudding', 'study']
    max_distance_km: 最大步行距離（預設 1.0 公里，約走路 15 分鐘）
    """
    # 讀取桃園咖啡廳資料庫
    df = pd.read_csv("cafes.csv")

    # A. 篩選距離：計算每間咖啡廳與使用者的距離
    df["distance"] = df.apply(
        lambda row: haversine(user_lat, user_lng, row["lat"], row["lng"]),
        axis=1,
    )
    # 只保留距離在指定範圍內的店家
    filtered_df = df[df["distance"] <= max_distance_km].copy()

    # B. 篩選標籤：必須符合所有被勾選的標籤（交集 AND 邏輯）
    for tag in selected_tags:
        filtered_df = filtered_df[filtered_df[tag] == 1]

    return filtered_df


# 3. 地圖視覺化函式
def generate_map(user_lat, user_lng, filtered_df):
    # 以使用者位置為地圖中心點
    mymap = folium.Map(location=[user_lat, user_lng], zoom_start=15)

    # 在地圖上標記「使用者目前的位置」（用紅色星星表示）
    folium.Marker(
        location=[user_lat, user_lng],
        popup="我的位置",
        icon=folium.Icon(color="red", icon="star"),
    ).add_to(mymap)

    # 在地圖上標記篩選出來的桃園咖啡廳（用藍色大頭針）
    for _, row in filtered_df.iterrows():
        # 計算步行時間（假設每分鐘走 70 公尺，1 公里約 14-15 分鐘）
        walking_time = round(row["distance"] * 1000 / 70)

        popup_text = f"<b>{row['name']}</b><br>距離：{row['distance']:.2f} km<br>步行約：{walking_time} 分鐘"

        folium.Marker(
            location=[row["lat"], row["lng"]],
            popup=popup_text,
            icon=folium.Icon(color="blue", icon="coffee", prefix="fa"),
        ).add_to(mymap)

    # 將地圖儲存為網頁檔，可以用瀏覽器直接打開
    mymap.save("cafe_result_map.html")
    print("地圖已成功生成！請打開 cafe_result_map.html 查看結果。")


# ─── 測試執行 ───
if __name__ == "__main__":
    # 假設使用者目前在元智大學正門口附近 (經緯度)
    my_lat = 24.9701
    my_lng = 121.2630

    # 假設使用者今天想找：布丁好吃 (pudding) 且 適合讀書 (study) 的咖啡廳
    my_tags = ["pudding", "study"]

    # 執行搜尋
    results = search_cafes(my_lat, my_lng, my_tags, max_distance_km=1.0)

    # 印出終端機結果
    print(f"幫您找到 {len(results)} 間符合條件的咖啡廳：")
    print(results[["name", "distance"]])

    # 產生成果互動地圖
    generate_map(my_lat, my_lng, results)

幫您找到 0 間符合條件的咖啡廳：
Empty DataFrame
Columns: [name, distance]
Index: []
地圖已成功生成！請打開 cafe_result_map.html 查看結果。
